In [50]:
import pandas as pd
from sklearn.metrics import confusion_matrix
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import make_scorer, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [51]:
data = pd.read_csv('../dataset/final_dataset_diff_f_L10.csv', parse_dates=['GAME_DATE'])
data = data.round(2)

In [52]:
condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
data_test = data_test.drop(columns=['GAME_DATE', 'gameId', 'A_teamId', 'H_teamId'])
data_train = data[~condition]
data_train = data_train.drop(columns=['GAME_DATE', 'gameId', 'A_teamId', 'H_teamId'])
X_train = data_train.drop('HOME_WON', axis=1)  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop('HOME_WON', axis=1)  # Fonctionnalités
y_test = data_test['HOME_WON']

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
# Définir les métriques de performance à calculer
scoring = {'accuracy': make_scorer(accuracy_score), 'f1': make_scorer(f1_score)}

,gameId,GAME_DATE,HOME_WON,H_teamId,A_teamId,ELO,ELO_PROB,NB_WIN_L10,PCT_3PT_L10,PCT_LANCER_FRANC_L10,...,estimatedNetRating_L10,estimatedOffensiveRating_L10,estimatedPace_L10,estimatedTeamTurnoverPercentage_L10,netRating_L10,offensiveRating_L10,pacePer40_L10,pace_L10,trueShootingPercentage_L10,turnoverRatio_L10
0,21300018,2013-10-31,1,1610612741,1610612752,-65.52,0.10,-0.1,0.04,-0.10,...,-24.10,-1.40,4.06,-4.50,-18.50,1.20,3.75,4.50,-0.06,-4.00
1,21300019,2013-10-31,1,1610612746,1610612744,0.52,0.28,-0.1,-0.18,-0.22,...,-36.80,-16.20,-2.46,1.74,-40.80,-13.90,-5.00,-6.00,-0.10,2.00
2,21300020,2013-11-01,1,1610612766,1610612739,-28.04,0.20,-0.1,0.07,-0.03,...,-23.00,-13.50,-4.58,-6.00,-19.60,-15.20,-1.25,-1.50,-0.08,-6.40
3,21300021,2013-11-01,1,1610612753,1610612740,-112.60,-0.04,0.0,-0.03,-0.20,...,-4.45,-2.05,6.75,1.31,-2.60,-0.60,4.95,5.94,0.01,1.75
4,21300022,2013-11-01,0,1610612764,1610612755,-16.70,0.24,-0.1,0.02,-0.08,...,-13.70,-7.20,-3.84,-0.21,-14.10,-7.70,-2.91,-3.50,-0.07,-0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11894,22301226,2023-12-08,0,1610612757,1610612742,-131.40,-0.09,-0.1,0.03,0.08,...,-6.61,-9.34,-1.92,5.22,-6.22,-8.72,-2.12,-2.54,-0.02,5.50
11895,22301227,2023-12-08,1,1610612738,1610612752,70.75,0.46,0.0,-0.03,-0.01,...,0.35,-3.46,1.57,1.43,-0.36,-4.31,1.57,1.88,0.01,1.36
11896,22301228,2023-12-08,0,1610612756,1610612758,25.74,0.35,0.1,0.02,0.09,...,4.31,1.14,-4.72,1.29,4.54,2.96,-5.12,-6.15,0.01,1.50
11897,22301229,2023-12-07,0,1610612749,1610612754,99.08,0.28,0.3,0.01,-0.01,...,8.28,1.09,-3.27,0.98,6.22,-1.94,-1.42,-1.70,-0.00,0.73


In [53]:
# Création du modèle PMC
def create_mlp(input_shape, num_classes):
    model = models.Sequential([
        layers.Dense(65, activation='relu', input_shape=input_shape),
        layers.Dense(65, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

In [54]:
# Création du modèle
model = create_mlp(input_shape=X_train[0].shape, num_classes=2)


In [55]:
# Compilation du modèle
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Entraînement du modèle
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

# Évaluation du modèle
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

Epoch 1/10
338/338 [==============================] - 1s 2ms/step - loss: 0.6332 - accuracy: 0.6399 - val_loss: 0.6232 - val_accuracy: 0.6434
Epoch 2/10
338/338 [==============================] - 0s 1ms/step - loss: 0.6210 - accuracy: 0.6555 - val_loss: 0.6254 - val_accuracy: 0.6489
Epoch 3/10
338/338 [==============================] - 0s 1ms/step - loss: 0.6200 - accuracy: 0.6556 - val_loss: 0.6191 - val_accuracy: 0.6480
Epoch 4/10
338/338 [==============================] - 0s 1ms/step - loss: 0.6204 - accuracy: 0.6535 - val_loss: 0.6334 - val_accuracy: 0.6380
Epoch 5/10
338/338 [==============================] - 0s 1ms/step - loss: 0.6206 - accuracy: 0.6582 - val_loss: 0.6189 - val_accuracy: 0.6543
Epoch 6/10
338/338 [==============================] - 0s 1ms/step - loss: 0.6192 - accuracy: 0.6555 - val_loss: 0.6249 - val_accuracy: 0.6380
Epoch 7/10
338/338 [==============================] - 0s 1ms/step - loss: 0.6182 - accuracy: 0.6580 - val_loss: 0.6219 - val_accuracy: 0.6398
Epoch 